In [1]:
import pandas as pd
# import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import time
import requests

def print_full(x):
    with pd.option_context('display.max_columns', None):
        display(x)
 

## Alpha Vantage - Fetching Data
* Retrieve from Alpha Vantage
* Grab 500 records each day

In [2]:
# Alpha Vantage API key: MQF6CNSC5LIK0DJQ
# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
#
# Documentation here:
# https://www.alphavantage.co/documentation/#

    
apikey = 'MQF6CNSC5LIK0DJQ'


# Functions related to gathering historical price data for lists of tickers
# This will be stored locally and added to on a (fairly) regular basis
# Will make a useful test dataset for prototyping methods for identifying companies to watch
def load_tickerlist():
    data = pd.read_csv('AV/listing_status.csv')
    return data[ data['assetType'] == 'Stock']['symbol'].to_list()

# For earnings slightly different method of generating the full tickerlist
# We have separately compiled a list of all tickers and number of prices for each - use this as the basis
def load_tickerlist_earnings():
    data = pd.read_csv('AV/tickers-number-of-prices.csv')
    return data['ticker'].to_list()

# For overview, and maybe the fundamentals, use the master prices list - because we can then remove all the unknowns
# If timestamp is NA then ticker was unknown - remove from ticker list
# Then use list(set (...)) to remove duplicates
def load_tickerlist_overview_original():
    m = pd.read_csv('AV/tickers-master-long.csv', usecols=['ticker','timestamp'])
    return list(set(m[ m['timestamp'].notna()]['ticker']))

# Update - use the overview file since this contains all the overview data so (in theory) all other data should be available...
def load_tickerlist_overview():
    m = pd.read_csv('AV/av-overview.csv', usecols=['ticker','LatestQuarter','Block'])
    valid = m[ (m['LatestQuarter'].notna()) & (m['Block'].isna()) ].set_index('LatestQuarter').sort_index()
    return list(set(valid['ticker']))

# Data maintenance for Alpha Vantage data
# Block a ticker from being used - for various reasons:
# 1 : no date supplied in Latest Quarter
# 2 : unusual character in symbol eg - or .
# 3 : 5-letter in NASDAQ, version of an existing 3-letter ticker
# 4 : 5-letter ticker in NASDAQ, version of an existing 4-letter ticker
# 5 : Not enough data - only 5 months or less
def applyBlocks(r):
    if (r['LatestQuarter'] == '0000-00-00'):
        return 1
    if ('-' in r.name) | ('.' in r.name):
        return 2
    if (r['Exchange'] == 'NASDAQ') & (len(r.name) > 4) & (r.name[0:3] in AV.overview.index):
        return 3
    if (r['Exchange'] == 'NASDAQ') & (len(r.name) > 4) & (r.name[0:4] in AV.overview.index):
        return 4
    if (len(AV.balanceq.loc[r.name]) < 5):
        return 5

# ...and apply the above function immediately on the overview data
#overview['Block'] = overview.apply(applyBlocks, axis=1)


def load_ticker(t):
    t = str.upper(t)
    url = 'https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY_ADJUSTED&symbol=' + t + '&apikey=' + apikey + '&datatype=csv'
    print("URL:", url)
    # Set up an unknown ticker object which will be returned in the event that this ticker is unknown
    unknown = pd.DataFrame([[t, pd.to_datetime('1999-01-01')]], columns=['ticker','timestamp'])
    # Timeout ticker returns blank values, which means this ticker will be fetched again in future
    timeout = pd.DataFrame([['timeout', pd.to_datetime('1999-01-01')]], columns=['ticker','timestamp'])
    
    # In case of any errors fetching the data - just return unknown so we ignore this item
    try:
        data = pd.read_csv(url)
        print(data.head())
    except:
        return unknown
    
    # First possible error, unknown ticker symbol
    # Response will contain the text 'Error'
    if (data.iloc[0,0].find('Error') > 0):
        return unknown

    # Text contains 'subscribe' this is the timeout message - no more until tomorrow
    if (data.iloc[0,0].find('subscribe') > 0):
        return 'timeout'
    
    try:
        data['ticker'] = t
        return data
    except:
        # two possibilities here - an empty dataset (ok, just empty) or a timeout (return all null, system will check again later)
        return timeout

# Re-use the earnings function for the other fundamental data
# URL: function=BALANCE_SHEET
# Keys: annualEarnings -> annualReports
def load_earnings(t):
    t = str.upper(t)
    url = 'https://www.alphavantage.co/query?function=CASH_FLOW&symbol=' + t + '&apikey=' + apikey
    print("URL:", url)

    # Set up an unknown ticker object which will be returned in the event that this ticker is unknown
    unknown = pd.DataFrame([t], columns=['ticker'])
    
    # In case of any errors fetching the data - just return unknown so we ignore this item
    try:
        r = requests.get(url)
        data = r.json()

    except:
        print('Error in get call')
        return unknown
    
    print(data.keys())
    
    # Timeout returns an Information text
    if ('Information' in data.keys()):
        return 'timeout'

    # ...so possible does a Note Information text
    if ('Note' in data.keys()):
        print(r)
        return 'timeout'
    
    
    # Other errors are probably due to some issue with this ticker - drop it
    if ('Error Message' in data.keys()):
        return unknown
    
    if ('annualReports' not in data.keys()):
        return unknown
    
    if (len(data['annualReports']) == 0):
        a = unknown
    else:
        a = pd.DataFrame(data['annualReports'])
        a['ticker'] = t
        a.replace(to_replace={'None':np.nan}, inplace=True)
        
    if (len(data['quarterlyReports']) == 0):
        q = unknown
    else:
        q = pd.DataFrame(data['quarterlyReports'])
        q['ticker'] = t
        q.replace(to_replace={'None':np.nan}, inplace=True)
        
    # Update: only deal with quarterly reports
    return q
    
def load_overview(t):
    t = str.upper(t)
    url = 'https://www.alphavantage.co/query?function=OVERVIEW&symbol=' + t + '&apikey=' + apikey
    print("URL:", url)

    # Set up an unknown ticker object which will be returned in the event that this ticker is unknown
    unknown = pd.DataFrame([[t, pd.to_datetime('1999-01-01')]], columns=['ticker','LatestQuarter'])
    
    # In case of any errors fetching the data - just return unknown so we ignore this item
    try:
        r = requests.get(url)
        data = r.json()

    except:
        print('Error in get call')
        print(r)
        return unknown
    
    print('Fields in response:', len(data.keys()))
    
    # Timeout returns an Information text
    if ('Information' in data.keys()):
        print(data)
        return 'timeout'

    # ...so possible does a Note Information text
    if ('Note' in data.keys()):
        print(data)
        return 'timeout'
    
    
    # Other errors are probably due to some issue with this ticker - drop it
    if ('Error Message' in data.keys()):
        return unknown

    if (len(data.keys()) == 0):
        return unknown
    
    a = pd.DataFrame.from_dict(data={t:data}, orient='index').drop(columns=['Description','Address']).rename(columns={'Symbol':'ticker'})
    a.replace(to_replace={'None':np.nan}, inplace=True)
        
    return a
    

def load_master():
    master = pd.read_csv('AV/tickers-master-long.csv', parse_dates=['timestamp'])
    return master

def save_master(m):
    m.drop_duplicates(subset=['ticker','timestamp'], inplace=True)
    m.set_index(['ticker','timestamp']).to_csv('AV/tickers-master-long.csv')

def load_master_overview():
    return pd.read_csv('AV/av-overview.csv', parse_dates=['LatestQuarter','DividendDate','ExDividendDate'])

def save_master_overview(m):
    m.drop_duplicates(subset=['ticker','LatestQuarter'], inplace=True)
    m.set_index(['ticker','LatestQuarter']).to_csv('AV/av-overview.csv')

def load_master_earnings():
    q = pd.read_csv('AV/av-earnings-quarterly.csv', parse_dates=['fiscalDateEnding'])
    return aq

def save_master_earnings(q):
    q.drop_duplicates(subset=['ticker','fiscalDateEnding'], inplace=True)
    q.set_index(['ticker','fiscalDateEnding']).to_csv('AV/av-earnings-quarterly.csv')

def load_master_bs():
    q = pd.read_csv('AV/av-balancesheet-quarterly.csv', parse_dates=['fiscalDateEnding'])
    return q

def save_master_bs(q):
    q.drop_duplicates(subset=['ticker','fiscalDateEnding'], inplace=True)
    q.set_index(['ticker','fiscalDateEnding']).to_csv('AV/av-balancesheet-quarterly.csv')
    
def load_master_cf():
    q = pd.read_csv('AV/av-cashflow-quarterly.csv', parse_dates=['fiscalDateEnding'])
    return q

def save_master_cf(q):
    q.drop_duplicates(subset=['ticker','fiscalDateEnding'], inplace=True)
    q.set_index(['ticker','fiscalDateEnding']).to_csv('AV/av-cashflow-quarterly.csv')
    
def load_master_is():
    q = pd.read_csv('AV/av-incomestatement-quarterly.csv', parse_dates=['fiscalDateEnding'])
    return q

def save_master_is(q):
    q.drop_duplicates(subset=['ticker','fiscalDateEnding'], inplace=True)
    q.set_index(['ticker','fiscalDateEnding']).to_csv('AV/av-incomestatement-quarterly.csv')
    

def load_next_ticker(fulllist=np.nan, tickers_so_far=np.nan):
    if (fulllist is np.nan):
        fulllist = load_tickerlist()
    if (tickers_so_far is np.nan):
        tickers_so_far = pd.read_csv('tickers-master-long.csv')['ticker'].unique()
    main_list = np.setdiff1d(fulllist, tickers_so_far)
    print('Tickers remaining:', len(main_list))
    if (len(main_list) == 0):
        return 'DONE'
    return load_ticker(main_list[0])

def load_next_overview(fulllist=np.nan, tickers_so_far=np.nan):
    if (fulllist is np.nan):
        fulllist = load_tickerlist_overview()
    if (tickers_so_far is np.nan):
        tickers_so_far = pd.read_csv('AV/av-overview.csv')['ticker'].unique()
    print('Full list:', len(fulllist), '  Tickers so far:', len(tickers_so_far))
    try:
        item = fulllist.pop(0)
        while (item in tickers_so_far):
            item = fulllist.pop(0)
    except:
        print('*** DONE! ***')
        return "DONE"
    return load_overview(item)

def load_next_earnings(fulllist, tickers_so_far):
    print('Full list:', len(fulllist), '  Tickers so far:', len(tickers_so_far))
    try:
        item = fulllist.pop(0)
        while (item in tickers_so_far):
            item = fulllist.pop(0)
    except:
        print('*** DONE! ***')
        return "DONE"
    print('Full list:', len(fulllist), '  Tickers so far:', len(tickers_so_far))
    return load_earnings(item)

def validate_earnings():
    q = load_master_earnings()
    qc = q.groupby('ticker')['fiscalDateEnding'].count()
    single = []
    for i in (set(qc.index) & set(ac.index)):
        if ((qc[i] == 1) & (ac[i] == 1)):
            single.append(i)
    print('Unknowns:',single)
            
    
# add_batch_prices: performs ALL the work required to grab some more data
# including reading all the relevant files and saving again...
# TODO: modify to only update the master file to save continually re-writing entire file each time
# Should be easy to simply add lines as needed, nothing ever deleted
def add_batch_prices(r=100, r2=5):
    master = load_master()
    alltickers = load_tickerlist()
    for i in range(0,r):
        for j in range(0,r2):
            next_ticker = load_next_ticker(alltickers, master['ticker'].unique() )
            if (type(next_ticker) is str):
                break
            master = pd.concat([master,next_ticker])
        # if not last time through loop then sleep (prevents unneccesary minute wait at the end)
        if (i+1 < r):
            time.sleep(60)
    # moved this line to the end to save poor hard drive from continually re-writing large amounts of data
    # only problem could be if any errors occur before this point the file won't be saved, but it seems quite robust now...
    save_master(master)

# Consider changing this from earnings to general report (useful for all the fundamentals)
# Try with just BALANCE_SHEET and see how it goes...
# Load the tickerlist_overview rather than tickerlist_earnings - uses the master tickerlist to remove unknowns
def add_batch_earnings(r=100, r2=5):
    masterq = load_master_cf()
    alltickers = load_tickerlist_overview()
    for i in range(0,r):
        for j in range(0,r2):
            q = load_next_earnings(alltickers, masterq['ticker'].unique() )
            if (type(q) is str):
                break
            masterq = pd.concat([masterq, q])
        # Extra step - occasionally save master along the way, useful when doing very large updates...
        if ((i>0) & (i % 101 == 0)):
            print('*** SAVING MASTER ***')
            save_master_cf(masterq)
        # if not last time through loop then sleep (prevents unneccesary minute wait at the end)
        if (i+1 < r):
            time.sleep(60)
    # moved this line to the end to save poor hard drive from continually re-writing large amounts of data
    # only problem could be if any errors occur before this point the file won't be saved, but it seems quite robust now...
    save_master_cf(masterq)

def add_batch_overview(r=100, r2=5):
    master = load_master_overview()
    alltickers = load_tickerlist_overview()
    for i in range(0,r):
        for j in range(0,r2):
            a = load_next_overview(alltickers, master['ticker'].unique() )
            if (type(a) is str):
                break
            master = pd.concat([master, a])
        # Extra step - occasionally save master along the way, useful when doing very large updates...
        if ((i>0) & (i % 101 == 0)):
            print('*** SAVING MASTER ***')
            save_master_overview(master)
        # if not last time through loop then sleep (prevents unneccesary minute wait at the end)
        if (i+1 < r):
            time.sleep(60)
    # moved this line to the end to save poor hard drive from continually re-writing large amounts of data
    # only problem could be if any errors occur before this point the file won't be saved, but it seems quite robust now...
    save_master_overview(master)

    

In [16]:
masterq = load_master_cf()
masterq
alltickers = load_tickerlist_overview()
alltickers

# Test it...
#add_batch_earnings(5,1)
print('Number of tickers:', len(alltickers))

master = load_master_overview()
master
add_batch_overview(3,3)

Number of tickers: 6280
Full list: 6280   Tickers so far: 9071
*** DONE! ***
Full list: 0   Tickers so far: 9071
*** DONE! ***
Full list: 0   Tickers so far: 9071
*** DONE! ***


### Documentation


In [4]:
# Some attempt to rationalise the AV historical data and figure out what's missing from where and how much
#
# Master list: listing_status / av_overview
# Note these are TWO separate API callls in AV even though I could combine
# overview provides latest financial data plus other useful 'header' info eg sector, industry
# listing_status shows if stock still listed on an exchange, date is was listed/delisted - not sure how useful this is?
# This provides the long list of all tickers I could use
# TODO: make this more useful and use as general index to store additional meta data for each stock (eg date last updated, 'interest' score)
# Also grab this file from AV periodically to ensure it remains reasonably uptodate (yearly?)

# Overview: av-overview.csv
# This is a general 'header' row for each company - should maybe replace the above listing_status since it holds a record for each company and I can include 'last update' and others here

# Prices list: tickers-master-long.csv
# Differently named because I started with this one - price data

# Earnings list: earnings
# Not sure how useful this is - its basically just the earnings - if I just grab each quarter earnings reports?
# One extra is the estimatedEPS and the Surprise% - did they beat expectations?

# Fundamentals: balancesheet incomestatement cashflow
# These should all match each other and hopefully the prices and earnings


In [5]:
# Investigate the above files in more details and start trying to do this a bit more professionally - come on Ben...

# Overview - how useful is this?
ov = pd.read_csv('AV/av-overview.csv')
ov

# listing status ?
status = pd.read_csv('AV/listing_status.csv')
status['status'].value_counts()
status['assetType'].value_counts()

status['exchange'].value_counts()


exchange
NASDAQ       5366
NYSE         3185
NYSE ARCA    2021
BATS          635
NYSE MKT      355
Name: count, dtype: int64

In [6]:
# Do we need both?
ov[ ov['Country'] == 'China']
ov['Industry'].value_counts()
ov['Sector'].value_counts()


Sector
REAL ESTATE & CONSTRUCTION    2624
LIFE SCIENCES                 1422
TECHNOLOGY                    1272
FINANCE                        940
TRADE & SERVICES               893
MANUFACTURING                  848
ENERGY & TRANSPORTATION        685
Name: count, dtype: int64

In [7]:
# The AV list of Sectors and Industries does not look so complete - maybe the EOD list is more complete? Could use EOD just for the master list?
eod = pd.read_csv('EOD/US_LIST_OF_SYMBOLS.csv')
eod['Exchange'].value_counts()

eod[ eod['Exchange'].isin(['NASDAQ','NYSE'])]['Type'].value_counts()

Type
Common Stock       7984
Preferred Stock     658
ETF                 475
FUND                416
Mutual Fund           2
Name: count, dtype: int64

In [8]:
# Perform some checking on the current source data
universe = load_tickerlist()

prices = pd.read_csv('AV/tickers-master-long.csv', usecols=['ticker'])['ticker'].unique()

#overview = pd.read_csv('av-overview.csv', usecols=['ticker'])['ticker'].unique()
overview = load_tickerlist_overview()
bs = pd.read_csv('AV/av-balancesheet-quarterly.csv', usecols=['ticker'])['ticker'].unique()
inc = pd.read_csv('AV/av-incomestatement-quarterly.csv', usecols=['ticker'])['ticker'].unique()
cf = pd.read_csv('AV/av-cashflow-quarterly.csv', usecols=['ticker'])['ticker'].unique()

In [9]:
print('In Universe NOT in prices:', list(set(universe) - set(prices)))
print()
print('In Universe NOT in overview:', list(set(universe) - set(overview)))
print()
print('In prices NOT in universe:', list(set(prices) - set(universe)))
print()
print('In prices NOT in overview:', list(set(prices) - set(overview)))
print()
print('In overview NOT in prices:', list(set(overview) - set(prices)))
print('In overview NOT in balance sheet:', list(set(overview) - set(bs)))
print('In overview NOT in income:', list(set(overview) - set(inc)))
print('In overview NOT in cashflow:', list(set(overview) - set(cf)))


print( f'Total universe {len(list(set(universe)))} total overview: {len(list(set(overview)))} total prices: {len(list(set(prices)))}' )
print( f'Total balance sheet {len(list(set(bs)))} total income: {len(list(set(inc)))} total cashflow: {len(list(set(cf)))}' )


In Universe NOT in prices: ['DFLI', 'BUJAU', 'SFR', 'SYM', 'OCTO', 'ZLSWU', 'RSF-R', 'MAMA', 'WLGS', 'STIXW', 'ACNT', 'VTS', 'FLFVU', 'BLACR', 'ELV', 'BNRG', 'CNXA', 'LIFWW', 'OSAAW', 'IPXXU', 'EFHTR', 'EHAB', 'ATH-P-E', 'TBMCU', 'ECX', 'BOCNU', 'HIVE', 'NXT(EXP20091224)', 'PSNYW', 'RITM-P-D', 'AUGT', 'SCWO', 'SAZ', 'BOF', 'INLX', 'PNACW', 'ZFOXW', 'PAVS', 'INAQW', 'SCCF', 'BLACW', 'SFRWW', 'IRAA', 'ZAPPW', 'EP', 'TLSIW', 'HUBCZ', 'BGXX', 'DMYY', 'CRBG', 'FGMCW', 'CZFS', 'GSUN', 'MGOL', 'CUBT', 'RWT-P-A', 'AIU', 'HMACR', 'EPOW', 'ILLM', 'CVKD', 'KNF-W', 'CHRD', 'MOB', 'ATMVU', 'LSDI', 'MLECW', 'CLMB', 'BIAF', 'YOSH', 'MRT', 'SHLT', 'INDV', 'EMCGW', 'JNJ-WD', 'SOUNW', 'STHO', 'BOAC-U', 'LNZA', 'BTMDW', 'SFWL', 'AMPX', 'GSDWW', 'CRGO', 'TOROV', 'BOWNR', 'ISRL', 'SAY', 'ACON', 'NAAS', 'SATX', 'RUMBW', 'MNK', 'RWAYZ', 'QDROW', 'SPGC', 'AGAE', 'CARM', 'MLYS', 'BENFW', 'MEDX', 'GRI', 'LIPO', 'SEPAW', 'AULT-P-D', 'JCSE', 'MGRM', 'ALAR', 'CCLDP', 'MLGO', 'LFCR', 'KEY-P-L', 'ALPS', 'FSCO', 'MCO

### Question: how come prices has more than UNIVERSE and OVERVIEW ?
* Seems that I got hold of more prices than the master lists hold... ???
* Hard to work with this without some idea of the total list I'm dealing with...

### Annual v Quarterly
* Is it possible to accurately create the annual from the quarterly?
* Generate annual from totalling quarterly and see how they compare


* Conclusion: YES - quarterly can also generate annual (upshot: annual not really needed better to use rolling quarterly)

In [10]:
# NOTE: this cell won't even run anymore since the function has changed to only deal with quarterly...
# a,q = load_master_earnings()

# # Choose a ticker at random...
# ticker = 'MNST'
# axq = q.set_index(['ticker','fiscalDateEnding']).loc[ticker].unstack(level=0)['reportedEPS'].sort_index()
# axa = a.set_index(['ticker','fiscalDateEnding']).loc[ticker].unstack(level=0)['reportedEPS'].sort_index()

# # Rolling window with a sum
# derived = axq.rolling(window=4).sum()
# derived

# # Reconcile the two versions
# rec = pd.DataFrame(data={'annual':axa, 'calculated':derived})
# rec[ rec['annual'].notna() ]

In [11]:
# Functions for analysing and further processing of historical price data
# All functions can assume a data format based on the master ticker data
# This means we can build on the data shape to generate results for all tickers in one pass

# NOTE: this cell and future cells should be put into a new notebook this notebook for collecting data ONLY
def percentage_gain(s):
    #print(s)
    last = s.last_valid_index()
    first = s.first_valid_index()
    if (last is None):
        return -1
    total_return = ( s.loc[last] - s.loc[first] ) / s.loc[first]
    #print("Total return:\n" , total_return)
    ndays = (pd.to_datetime(last) - pd.to_datetime(first)).days
    # initialise annualized to -1 which will be returned for low numbers of days
    annualized = -1
    if (ndays > 720):
        annualized = (1 + total_return)** (365/ndays) - 1
        #print("Number of days:", ndays)
        #print("Annualized:\n", annualized)
    return annualized

# calc_cum_prod: generates a single column dataframe with all tickers as the index and the cum_prod for each
# allows for a quick check of best cumulative returns
def calc_cum_prod(m):
    return m.apply(percentage_gain, axis=0)

In [12]:
# I believe close and adjusted close will always be the same on todays' date
# It is only historical prices which are adjusted to take into account new information - to make historical comparison better
#
# Use adjusted close to ensure consistency, when looking at historical data
#
# Quick fun check - which has the highest cumulative return of all stocks?
#master = load_master()
close = master.set_index(['ticker','timestamp']).unstack(level=0)['adjusted close']
ret = (1+close.pct_change()).cumprod()
ret

NameError: name 'master' is not defined

In [ ]:
final = ret.iloc[-1,:].sort_values(ascending=False)
top10 = final[2:12].index.to_list()
top10

In [ ]:
plt.figure(figsize=(12,8))
plt.plot(ret[top10])
plt.legend()

# Adjust the width of the legend lines
for leg_line in plt.legend().get_lines():
    plt.setp(leg_line, linewidth=6)

#ret[top10]

In [ ]:
recent_close = close.loc[ '2017':]
recent_close

pct_change = recent_close.pct_change()
cum_prod = (1+pct_change).cumprod()
results = cum_prod.iloc[-1].sort_values(ascending=False)
results.head(20)

In [ ]:
recent_close['SKE'].dropna().plot()

In [ ]:
master.set_index(['ticker','timestamp']).loc['APPS'].sort_index()